In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# ----------------------------
# Paths
# ----------------------------
images_folder = Path("/data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs")  
labels_folder = Path("/data/colon_cancer/Classifier/Decathlon/raw_splitted/labelsTs")  
output_csv = Path("/data/colon_cancer/Classifier/Decathlon/labels.csv") 
"""
for img_path in images_folder.glob("*0000.nii.gz"): 
    # Extract number from filename
    number = img_path.name.replace("_0000.nii.gz", "")
    number_int = int(number)  # convert to int to remove leading zeros
    
    new_name = f"._colon_{number_int}.nii.gz"  # e.g., 12_0000.nii
    new_path = images_folder / new_name
    shutil.move(str(img_path), str(new_path))
"""


# ----------------------------
# Rename images
# ----------------------------
renamed_images = []

for img_path in images_folder.glob("colon_*.nii.gz"):
    # Extract number from filename
    number = img_path.name.replace("colon_", "").replace(".nii.gz", "")  
    number_int = int(number)  # convert to int to remove leading zeros
    
    new_name = f"{number_int}_0000.nii.gz"  # e.g., 12_0000.nii
    new_path = images_folder / new_name
    shutil.move(str(img_path), str(new_path))
    
    renamed_images.append(new_path)

# ----------------------------
# Rename labels
# ----------------------------
for lbl_path in labels_folder.glob("colon_*.nii.gz"):
    number = lbl_path.name.replace("colon_", "").replace(".nii.gz", "")  # '001' from 'colon_001'
    number_int = int(number)  # convert to int
    new_name = f"{number_int}.nii.gz"
    new_path = labels_folder / new_name
    shutil.move(str(lbl_path), str(new_path))

# ----------------------------
# Generate CSV
# ----------------------------
data = []

for img_path in sorted(images_folder.glob("*_0000.nii.gz")):
    uid = img_path.stem.split("_")[0]  # get the UID (number before '_0000')
    data.append({
        "UID": uid,
        "img_path": str(img_path),
        "target": 1,
        "Split": "test",
        "Fold": 0
    })

df = pd.DataFrame(data, columns=["UID", "img_path", "target", "Split", "Fold"])
df.to_csv(output_csv, index=False)
print(f"CSV saved to {output_csv}")


In [1]:
import pandas as pd
import numpy as np

# -------------------------
# Config
# -------------------------
INPUT_CSV = "/data/colon_cancer/Classifier/ColonCancer/labels.csv"
OUTPUT_CSV = "/data/colon_cancer/Classifier/ColonCancer/splits_new.csv"

N_TRAIN = 628
N_VAL   = 125
N_TEST  = 80

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# -------------------------
# Load data
# -------------------------
df = pd.read_csv(INPUT_CSV)

assert len(df) == N_TRAIN + N_VAL + N_TEST, "Split sizes do not sum to dataset size!"

# -------------------------
# Compute class proportions
# -------------------------
class_counts = df["target"].value_counts().sort_index()
total = len(df)

class_ratios = class_counts / total

# Samples per class per split
def split_counts(n_total):
    counts = (class_ratios * n_total).round().astype(int)
    # fix rounding errors
    diff = n_total - counts.sum()
    if diff != 0:
        counts.iloc[0] += diff
    return counts

train_counts = split_counts(N_TRAIN)
val_counts   = split_counts(N_VAL)
test_counts  = split_counts(N_TEST)

# -------------------------
# Perform stratified split
# -------------------------
df["split"] = None
remaining_idx = []

for cls in class_counts.index:
    cls_df = df[df["target"] == cls].sample(frac=1, random_state=RANDOM_SEED)

    n_train = train_counts[cls]
    n_val   = val_counts[cls]
    n_test  = test_counts[cls]

    train_idx = cls_df.iloc[:n_train].index
    val_idx   = cls_df.iloc[n_train:n_train + n_val].index
    test_idx  = cls_df.iloc[n_train + n_val:n_train + n_val + n_test].index

    df.loc[train_idx, "split"] = "train"
    df.loc[val_idx, "split"]   = "val"
    df.loc[test_idx, "split"]  = "test"

# -------------------------
# Sanity checks
# -------------------------
print("\nSplit sizes:")
print(df["split"].value_counts())

print("\nClass balance per split:")
print(df.groupby(["split", "target"]).size().unstack())

assert df["split"].isna().sum() == 0, "Some samples were not assigned a split!"

# -------------------------
# Save
# -------------------------
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved splits to {OUTPUT_CSV}")



Split sizes:
split
train    628
val      125
test      80
Name: count, dtype: int64

Class balance per split:
target    0    1
split           
test     32   48
train   251  377
val      50   75

Saved splits to /data/colon_cancer/Classifier/ColonCancer/splits_new.csv
